In [1]:
from CountsinCylinders import *
from astropy.table import Table
from astropy.table import vstack
import fitsio

In [2]:
#set up auto-reload for development purposes

%load_ext autoreload
%autoreload 2

In [3]:
#general parameters
R_CiC = 1 #radius in Mpc
L_CiC = 40 #height of cylinder in Mpc
zlim = [0.8,1]
psilim = [0.2,1.5]

## Compute CiC in SV3 ##

In [ ]:
#loadholes
holetableselg = []
holetableslrg = []
for i in range(117):
    spelg = Table.read("datafiles/Holes_ELG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/Holes_LRG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))

In [ ]:
#SV3 CiC, 26 July 2025
#zlim = [0,1] #just for debugging 2pcf comparison
#FINAL for paper!!
#elgdownsamplefrac = 0.9627664301054767
#LSS SV3 catalogs, using ELGnotqso
file_elgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_S_clustering.dat.fits'#'/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_S_clustering.dat.fits'
file_elgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_N_clustering.dat.fits'#'/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_N_clustering.dat.fits'
file_lrgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_S_clustering.dat.fits'
file_lrgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_N_clustering.dat.fits'
columns = ['RA','DEC','Z']
columnnames = columns[:3]
sp1 = Table(fitsio.read(file_elgN, columns=columns))
sp2 = Table(fitsio.read(file_elgS, columns=columns))
sp3 = Table(fitsio.read(file_lrgN, columns=columns))
sp4 = Table(fitsio.read(file_lrgS, columns=columns))
spE = vstack([sp1,sp2])
spL = vstack([sp3,sp4])
spelg = spE[(spE["Z"] > zlim[0])&(spE["Z"] < zlim[1])]
splrg = spL[(spL["Z"] > zlim[0])&(spL["Z"] < zlim[1])]


primarymask = makePrimariesRosettes
primaryargs = [*zlim,*psilim,R_CiC,L_CiC,'RA','DEC']

outfilename = "SV3_z0.8_1.6_R"+str(R_CiC)+"L40"

In [ ]:

R_hole = 0.03*np.pi/180 #convert from degrees to radians
countsInCylindersHoleClean(spelg, splrg, holetableelg,holetablelrg,columnnames, R_CiC, L_CiC,R_hole, zlim, primarymask, primaryargs, outfilename)
#countsInCylinders(spelg, splrg, columnnames, R_CiC, L_CiC, zlim, primarymask, primaryargs, outfilename, secondarymask = makePrimariesRosettesNoZBoundary,secondaryargs=primaryargs)

In [ ]:
#there is actually only one complete catalog
mockfile = "/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/mockTargets_000_FirstGen_CutSky_alltracers_sv3bits.fits"

In [ ]:
#To check completeness in SV3
zlim = [0.75,1] #just for debugging 2pcf comparison

for i in range(25):
    mockdir = "/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/LSS_rea"
    mockdir += "{:03d}".format(i)
    mockdir += "/fuji/LSScats/1/"

    felgcomplete = mockdir + "ELG_full.dat.fits"
    felgincomplete = mockdir + "ELG_clustering.dat.fits"
    flrgcomplete = mockdir + "LRG_full.dat.fits"
    flrgincomplete = mockdir + "LRG_clustering.dat.fits"
    
    columns = ['RA','DEC','Z']
    elgcomplete = Table(fitsio.read(felgcomplete, columns=columns))
    elgincomplete = Table(fitsio.read(felgincomplete, columns=columns))
    lrgcomplete = Table(fitsio.read(flrgcomplete, columns=columns))
    lrgincomplete = Table(fitsio.read(flrgcomplete, columns=columns))
    
    elgcomplete
    primarymask = makePrimariesRosettes
    primaryargs = [*zlim,*psilim,R_CiC,L_CiC,'RA','DEC']
    
    outfile_complete = "SV3_mock_complete_"+str(i)
    outfile_incomplete = "SV3_mock_incomplete_"+str(i)
    R_hole = 0.03*np.pi/180 #convert from degrees to radians

    #complete
    countsInCylindersHoleClean(elgcomplete, lrgcomplete, holetableelg,holetablelrg,columnnames, R_CiC, L_CiC,R_hole, zlim, primarymask, primaryargs, outfile_complete)

    #incomplete
    countsInCylindersHoleClean(elgincomplete, lrgincomplete, holetableelg,holetablelrg,columnnames, R_CiC, L_CiC,R_hole, zlim, primarymask, primaryargs, outfile_incomplete)


    

In [ ]:
#rosettes['n_elim'] = [273,223,188,224,194,247,266,170,259,287,267,359,326,258,382,413,175,227,373262]

### Counts in Cylinders in Abacus ##

In [ ]:
#attempt to clean the forFA mocks
elgmocks = []
lrgmocks = []
columns = ['RA','DEC','RSDZ']
columnnames = columns[:4]
primarymask = makePrimariesZandAngular
ralim = [130,223]
declim  = [-5.25,3.75]
primaryargs = [*zlim,*ralim,*declim,R_CiC,L_CiC]
#zfailratio = 0.04649021864211737
print('check')

#loadholes
holetableselg = []
holetableslrg = []
for i in range(240):
    spelg = Table.read("datafiles/forFAelg_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/forFAlrg_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))

In [ ]:
R_hole = 0.03*np.pi/180 #convert from degrees to radians



#forFA Mocks, downsampled by redshift fraction but NOT using ebits
mocks = []
columns = ['RA','DEC','RSDZ','ZWARN','DESI_TARGET','MASKBITS'] #want to 
columnnames = columns[:3]
primarymask = makePrimariesZandAngular
ralim = [130,223]
declim  = [-5.25,3.75]
primaryargs = [*zlim,*ralim,*declim,R_CiC,L_CiC]

print(zlim)
for mocknumber in range(18):
    #for the moment, don't downsample by redshift success fraction
    lrgsuccess = 1 #0.90 #0.976*0.8812216895357733
    elgsuccess = 0.8062 #1 #0.811 #1 #0.97 #0.8360495775774505   #1 #0.86 #0.7171793765301621 #0.69/0.9627664301054767
    outfilename = "ffa_0.8_1.6_"+str(mocknumber)+"_R"+str(R_CiC)+"L40"
    mockdir = mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    #mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    file = mockdir+ "forFA"+str(mocknumber)+".fits"
    #file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
    #file_lrg = mockdir + "mock"+str(mocknumber)+"/LRG_complete_clustering.dat.fits"
    #get lrg mask from "matched input" lrg file
    sp = Table(fitsio.read(file,columns=columns))
    #sp = Table(fitsio.read(file_lrg,columns=columns))
    splrg = sp[(sp["DESI_TARGET"]&0x1==0x1)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]
    #sp = Table(fitsio.read(file_elg,columns=columns))
    #file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
    #spelg = Table(fitsio.read(file_elg,columns=['RA','DEC','Z']))
    spelg = sp[(sp["DESI_TARGET"]&0x20==0x20)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]
    #spelg['RSDZ']=spelg['Z']
    elg_downsampled = spelg[[(np.random.rand() < elgsuccess) for i in range(len(spelg))]]
    lrg_downsampled = splrg[[(np.random.rand() < lrgsuccess) for i in range(len(splrg))]]
    countsInCylindersHoleClean(elg_downsampled,lrg_downsampled,holetablelrg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename)

In [ ]:
#compute CiC in mocks that have fiberassign

R_hole = 0.03*np.pi/180 #convert from degrees to radians



#forFA Mocks, downsampled by redshift fraction but NOT using ebits
mocks = []
columns = ['RA','DEC','Z','ZWARN','DESI_TARGET','MASKBITS'] #want to 
columnnames = columns[:3]
primarymask = makePrimariesZandAngular
ralim = [130,223]
declim  = [-5.25,3.75]
primaryargs = [*zlim,*ralim,*declim,R_CiC,L_CiC]

print(zlim)
for mocknumber in range(18):
    #for the moment, don't downsample by redshift success fraction
    lrgsuccess = 1 #0.90 #0.976*0.8812216895357733
    elgsuccess = 0.8062 #1 #0.811 #1 #0.97 #0.8360495775774505   #1 #0.86 #0.7171793765301621 #0.69/0.9627664301054767
    outfilename = "ffa_cleaned_"+str(mocknumber)+"_R"+str(R_CiC)+"L40"
    mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    #mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    #file = mockdir+ "mock"+str(mocknumber)+"/.fits"
    file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_ffa_clustering.dat.fits"
    file_lrg = mockdir + "mock"+str(mocknumber)+"/LRG_ffa_clustering.dat.fits"
    #get lrg mask from "matched input" lrg file
    #sp = Table(fitsio.read(file,columns=columns))
    sp = Table(fitsio.read(file_lrg,columns=columns))
    splrg = sp[(sp["Z"] > zlim[0])&(sp["Z"] < zlim[1])]
    sp = Table(fitsio.read(file_elg,columns=columns))
    #file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
    sp = Table(fitsio.read(file_elg,columns=['RA','DEC','Z']))
    spelg = sp[(sp["Z"] > zlim[0])&(sp["Z"] < zlim[1])]
    #spelg['RSDZ']=spelg['Z']
    elg_downsampled = spelg[[(np.random.rand() < elgsuccess) for i in range(len(spelg))]]
    lrg_downsampled = splrg[[(np.random.rand() < lrgsuccess) for i in range(len(splrg))]]
    countsInCylindersHoleClean(elg_downsampled,lrg_downsampled,holetablelrg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename)

## Compare Fiberassign to no Fiberassign (Abacus) ##

In [ ]:
#compute CiC in mocks that do not have fiberassign

R_hole = 0.03*np.pi/180 #convert from degrees to radians



#forFA Mocks, downsampled by redshift fraction but NOT using ebits
mocks = []
columns = ['RA','DEC','Z','ZWARN','DESI_TARGET','MASKBITS'] #want to 
columnnames = columns[:3]
primarymask = makePrimariesZandAngular
ralim = [130,223]
declim  = [-5.25,3.75]
primaryargs = [*zlim,*ralim,*declim,R_CiC,L_CiC]

print(zlim)
for mocknumber in range(18):
    #for the moment, don't downsample by redshift success fraction
    lrgsuccess = 1 #0.90 #0.976*0.8812216895357733
    elgsuccess = 0.8062 #1 #0.811 #1 #0.97 #0.8360495775774505   #1 #0.86 #0.7171793765301621 #0.69/0.9627664301054767
    outfilename = "Abacus_complete_cleaned_"+str(mocknumber)+"_R"+str(R_CiC)+"L40"
    mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    #mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    #file = mockdir+ "mock"+str(mocknumber)+"/.fits"
    file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
    file_lrg = mockdir + "mock"+str(mocknumber)+"/LRG_complete_clustering.dat.fits"
    #get lrg mask from "matched input" lrg file
    #sp = Table(fitsio.read(file,columns=columns))
    sp = Table(fitsio.read(file_lrg,columns=columns))
    splrg = sp[(sp["Z"] > zlim[0])&(sp["Z"] < zlim[1])]
    sp = Table(fitsio.read(file_elg,columns=columns))
    #file_elg = mockdir + "mock"+str(mocknumber)+"/ELG_LOP_complete_clustering.dat.fits"
    sp = Table(fitsio.read(file_elg,columns=['RA','DEC','Z']))
    spelg = sp[(sp["Z"] > zlim[0])&(sp["Z"] < zlim[1])]
    #spelg['RSDZ']=spelg['Z']
    elg_downsampled = spelg[[(np.random.rand() < elgsuccess) for i in range(len(spelg))]]
    lrg_downsampled = splrg[[(np.random.rand() < lrgsuccess) for i in range(len(splrg))]]
    countsInCylindersHoleClean(elg_downsampled,lrg_downsampled,holetablelrg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename)

# Compare Fiberassign to No Fiberassign (SV3) #

In [4]:
#loadholes
holetableselg = []
holetableslrg = []
for i in range(117):
    spelg = Table.read("datafiles/Holes_ELG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/Holes_LRG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))

4646361 4697603


## With Fiberassign ##

In [5]:
#with fiberassign
#1/2/2026
file_elg = "/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/LSS_rea000/fuji/LSScats/1/ELG_clustering.dat.fits"
file_lrg ="/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/LSS_rea000/fuji/LSScats/1/LRG_full.dat.fits"
columns = ['RA','DEC','Z',"DESI_TARGET"]
columnnames = columns[:3]
spE = Table(fitsio.read(file_elg))#, columns=columns))
spL = Table(fitsio.read(file_lrg))#, columns=columns))
spelg = spE[(spE["Z"] > zlim[0])&(spE["Z"] < zlim[1])]#&(spE["SV3_DESI_TARGET"]&0x2000 == 0x2000)]
splrg = spL[(spL["Z"] > zlim[0])&(spL["Z"] < zlim[1])]


primarymask = makePrimariesRosettes
primaryargs = [*zlim,*psilim,R_CiC,L_CiC,'RA','DEC']

outfilename = "SV3_fiberassign"

In [6]:
len(spelg)

116826

In [154]:
file_tile = "/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/LSS_rea000/tiles-DARK.fits"
tbtile = Table(fitsio.read(file_tile))#, columns=columns))

In [7]:
R_hole = 0.03*np.pi/180 #convert from degrees to radians
countsInCylindersHoleClean(spelg, splrg, holetableelg,holetablelrg,columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename)
#countsInCylinders(spelg, splrg, columnnames, R_CiC, L_CiC, zlim, primarymask, primaryargs, outfilename, secondarymask = makePrimariesRosettesNoZBoundary,secondaryargs=primaryargs)

finding unit sphere coordinates of galaxies
finding Cartesian coordinates of galaxies
making cKDTrees
eliminating galaxies near holes
initial ELGs: 94164
initial primary LRGs: 33708
finished hole cleaning
ELG primaries remaining: 82630
LRG primaries remaining: 28282
Pass rate, ELGs: 0.8775115755490421
Pass rate, LRGs: 0.8390293105494244
Searching for neighbors in spheres
Pruning galaxies from spheres
Pruning ELG auto-correlation
   Galaxy 0
Pruning LRG auto-correlation
   Galaxy 0
Pruning ELG-LRG cross-correlation
   Galaxy 0
Pruning LRG-ELG cross-correlation
   Galaxy 0
Saved file datafiles/SV3_fiberassign_elg.csv
Saved file datafiles/SV3_fiberassign_lrg.csv


In [12]:
#no fiberassign
#1/2/2026
file = "/global/cfs/cdirs/desi/survey/catalogs/SV3/mocks/mockTargets_000_FirstGen_CutSky_alltracers_sv3bits.fits"
columns = ['RA','DEC','RSDZ','SV3_DESI_TARGET']
columnnames = columns[:3]
sp = Table(fitsio.read(file))

splrg = sp[(sp["SV3_DESI_TARGET"]&0x1==0x1)&(sp["ZWARN"]==0)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]
spelg = sp[(sp["SV3_DESI_TARGET"]&0x20==0x20)&(sp["ZWARN"]==0)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]

elgsuccess = 1 #0.9621
lrgsuccess = 1 #0.9883
elg_downsampled = spelg[[(np.random.rand() < elgsuccess) for i in range(len(spelg))]]
lrg_downsampled = splrg[[(np.random.rand() < lrgsuccess) for i in range(len(splrg))]]

primarymask = makePrimariesRosettes
primaryargs = [*zlim,*psilim,R_CiC,L_CiC,'RA','DEC']

outfilename = "SV3_nofiberassign"

In [13]:
R_hole = 0.03*np.pi/180 #convert from degrees to radians
countsInCylindersHoleClean(elg_downsampled, lrg_downsampled, holetableelg,holetablelrg,columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename)
#countsInCylinders(spelg, splrg, columnnames, R_CiC, L_CiC, zlim, primarymask, primaryargs, outfilename, secondarymask = makePrimariesRosettesNoZBoundary,secondaryargs=primaryargs)

finding unit sphere coordinates of galaxies
finding Cartesian coordinates of galaxies
making cKDTrees
eliminating galaxies near holes
initial ELGs: 98073
initial primary LRGs: 34061
finished hole cleaning
ELG primaries remaining: 85882
LRG primaries remaining: 28561
Pass rate, ELGs: 0.8756946356285624
Pass rate, LRGs: 0.8385249992660228
Searching for neighbors in spheres
Pruning galaxies from spheres
Pruning ELG auto-correlation
   Galaxy 0
Pruning LRG auto-correlation
   Galaxy 0
Pruning ELG-LRG cross-correlation
   Galaxy 0
Pruning LRG-ELG cross-correlation
   Galaxy 0
Saved file datafiles/SV3_nofiberassign_elg.csv
Saved file datafiles/SV3_nofiberassign_lrg.csv


# Counts around randoms #

## Abacus ##

In [ ]:
#general parameters
R_CiC = 1 #radius in Mpc
L_CiC = 40 #height of cylinder in Mpc
zlim = [0.75,1]
psilim = [0.2,1.5]

R_hole = 0.03*np.pi/180 #convert from degrees to radians
lrgsuccess = 1 #0.976*0.8812216895357733
elgsuccess = 0.8062 #0.8360495775774505 #0.7171793765301621 #0.69/0.9627664301054767
#forFA Mocks, downsampled by redshift fraction but NOT using ebits
mocks = []
columns = ['RA','DEC','RSDZ','ZWARN','DESI_TARGET','MASKBITS'] #want to 
colsrans = ['RA','DEC','Z']
columnnames = columns[:3]
primarymask = makePrimariesZandAngular
ralim = [130,223]
declim  = [-5.25,3.75]
primaryargs = [*zlim,*ralim,*declim,R_CiC,L_CiC]

#loadholes
holetableselg = []
holetableslrg = []
for i in range(240):
    spelg = Table.read("datafiles/forFAelg_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/forFAlrg_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))



for mocknumber in range(18):
    outfilename = "forFA_countsaroundrandoms_"+str(mocknumber)+"R_"+str(R_CiC)
    mockdir = "/global/cfs/cdirs/desi//survey/catalogs/Y1/mocks/SecondGenMocks/AbacusSummit_v4_2/"
    file = mockdir+ "forFA"+str(mocknumber)+".fits"
    directory = "/pscratch/sd/c/crmiller/"
    elgtabs = []
    lrgtabs = []
    for i in range(1):
        elgranfile = mockdir+"/mock"+str(mocknumber)+"/ELG_LOP_complete_12_clustering.ran.fits"
        lrgranfile = mockdir+"/mock"+str(mocknumber)+"/LRG_complete_12_clustering.ran.fits"
        spranelg = Table(fitsio.read(elgranfile,columns=colsrans))
        spranlrg = Table(fitsio.read(lrgranfile,columns=colsrans))
        spranelg = spranelg[(spranelg["Z"] > zlim[0])&(spranelg["Z"] < zlim[1])&(spranelg['RA'] > ralim[0])& \
            (spranelg['RA'] < ralim[1])&(spranelg['DEC'] > declim[0])&(spranelg['DEC'] < declim[1])]
        spranlrg = spranlrg[(spranlrg["Z"] > zlim[0])&(spranlrg["Z"] < zlim[1])&(spranlrg['RA'] > ralim[0])& \
            (spranlrg['RA'] < ralim[1])&(spranlrg['DEC'] > declim[0])&(spranlrg['DEC'] < declim[1])]
        elgtabs.append(spranelg)
        lrgtabs.append(spranlrg)
    spranselg = vstack(elgtabs)
    spranselg['RSDZ']=spranselg['Z']
    spranslrg = vstack(lrgtabs)
    spranslrg['RSDZ']=spranslrg['Z']
    sp = Table(fitsio.read(file,columns=columns))
    #get lrg mask from "matched input" lrg file
    splrg = sp[(sp["DESI_TARGET"]&0x1==0x1)&(sp["ZWARN"]==0)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]
    spelg = sp[(sp["DESI_TARGET"]&0x20==0x20)&(sp["ZWARN"]==0)&(sp["RSDZ"] > zlim[0])&(sp["RSDZ"] < zlim[1])]
    elg_downsampled = spelg[[(np.random.rand() < elgsuccess) for i in range(len(spelg))]]
    lrg_downsampled = splrg[[(np.random.rand() < lrgsuccess) for i in range(len(splrg))]]
    #we need separate ELG and LRG random catalogs
    #we count tracers of both types around each
    countsAroundRandomsHoleClean(spranslrg,elg_downsampled,lrg_downsampled,holetableelg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename+"lrg",directory=directory)
    countsAroundRandomsHoleClean(spranselg,elg_downsampled,lrg_downsampled,holetableelg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename+"elg_nodownsample",directory=directory)

## SV3 Counts around Randoms ##

In [ ]:
#loadholes
holetableselg = []
holetableslrg = []
for i in range(117):
    spelg = Table.read("datafiles/Holes_ELG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    splrg = Table.read("datafiles/Holes_LRG_SV3_tile"+str(i)+"_holes.csv",format = "csv")
    holetableselg.append(spelg)
    holetableslrg.append(splrg)
holetableelg = vstack(holetableselg)
holetablelrg = vstack(holetableslrg)
print(len(holetableelg),len(holetablelrg))


#Counts around randoms in SV3, 7/22/25
#FINAL for paper!!
zlim = [0.75, 1]
R_hole = 0.03*np.pi/180
#elgdownsamplefrac = 0.9627664301054767
#LSS SV3 catalogs, using ELGnotqso
file_elgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_S_clustering.dat.fits'
file_elgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_HIPnotqso_N_clustering.dat.fits'
file_lrgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_S_clustering.dat.fits'
file_lrgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_N_clustering.dat.fits'
columns = ['RA','DEC','Z']
columnnames = columns
sp1 = Table(fitsio.read(file_elgN, columns=columns))
sp2 = Table(fitsio.read(file_elgS, columns=columns))
sp3 = Table(fitsio.read(file_lrgN, columns=columns))
sp4 = Table(fitsio.read(file_lrgS, columns=columns))
spE = vstack([sp1,sp2])
spL = vstack([sp3,sp4])
spelg = spE[(spE["Z"] > zlim[0])&(spE["Z"] < zlim[1])]
splrg = spL[(spL["Z"] > zlim[0])&(spL["Z"] < zlim[1])]
primarymask = makePrimariesRosettes
primaryargs = [*zlim,*psilim,R_CiC,L_CiC,'RA','DEC']
outfilename = "SV3_countsaroundrandoms_"

ranelg = []
ranlrg = []
for i in range(18):
    file_elgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_S_'+str(i)+'_clustering.ran.fits'
    file_elgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/ELG_N_'+str(i)+'_clustering.ran.fits'
    file_lrgS = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_S_'+str(i)+'_clustering.ran.fits'
    file_lrgN = '/global/cfs/cdirs/desi/vac/edr/lss/v2.0/LSScats/clustering/LRG_main_N_'+str(i)+'_clustering.ran.fits'
    sp1 = Table(fitsio.read(file_elgN, columns=columns))
    sp2 = Table(fitsio.read(file_elgS, columns=columns))
    sp3 = Table(fitsio.read(file_lrgN, columns=columns))
    sp4 = Table(fitsio.read(file_lrgS, columns=columns))
    ranelg.append(sp1)
    ranelg.append(sp2)
    ranlrg.append(sp3)
    ranlrg.append(sp4)
spranselg = vstack(ranelg)
spranslrg = vstack(ranlrg)


countsAroundRandomsHoleClean(spranslrg,spelg,splrg,holetableelg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename+"lrg", directory = "/pscratch/sd/c/crmiller/")
countsAroundRandomsHoleClean(spranselg,spelg,splrg,holetableelg,holetablelrg,
                      columnnames, R_CiC, L_CiC, R_hole, zlim, primarymask, primaryargs, outfilename+"elg", directory = "/pscratch/sd/c/crmiller/")